# Notebook 4: DrugBank Screening

**Purpose:** Screen the DrugBank compound library against every trained,
calibrated, leak-free model this project produced for sensitivity analysis
(notebook 3 — all 5 targets x 3 activity pools x 3 feature representations),
while defining the PRIMARY prospective DrugBank screen strictly from each
target's deployed full-pool + combined-representation model. This produces
a ranked, uncertainty-aware primary candidate shortlist per target without
mixing endpoint-pool/model provenance. Kept as its own
notebook, separate from training, because training and screening both take
hours and are usually re-run independently of one another — screening can
be re-run (e.g. against an updated DrugBank release) without retraining
anything.

**No new models are trained here.** Every classifier, regressor, Platt
calibrator, conformal-prediction wrapper, and applicability-domain
reference this notebook uses was already fit and frozen by notebook 3
(Modules E/H) — this notebook only loads and applies them. This also means
the applicability-domain distances/status reported here are computed from
the SAME frozen kNN reference every other AD number in this project uses,
never refit (an earlier implementation issue — refitting AD from
scratch for screening — does not recur here).

**Required inputs:**
- A DrugBank structure-data file (`.sdf`), placed at the path set below.
  DrugBank's raw structure library is license-restricted and is not
  redistributed with this project — obtain it independently and set
  `DRUGBANK_SDF_PATH` before running.
- Notebook 3's outputs: `ml/models/{target}_{pool}/{representation}/` (deployed
  models, calibrators, conformal wrappers, AD references, feature names) and
  `ml/results/best_algorithm_by_combination.csv`.
- Notebook 2's outputs: `data/processed/cleaned_data_{target}_{pool}.csv`
  (for the "already in training set" cross-reference below).

**Generated outputs** (`ml/results/drugbank_screening/`, manifest +
SHA-256 hashes in `manifest_04_drugbank_screening.json`):
- Per (target, pool, representation) full screening table: calibrated
  probability, conformal prediction set/interval, applicability-domain
  status/distance, whether the compound was already in that target's
  training data.
- One consolidated PRIMARY ranked shortlist per target restricted to the
  deployed full activity pool + combined representation, with a
  high-confidence threshold.
- All-pool / all-representation screening outputs retained separately as
  sensitivity evidence; they do not contribute candidates to the primary
  prospective shortlist.
- Cross-target polypharmacology table — DrugBank compounds predicted active
  against more than one target.
- Drug-group (approved/investigational/experimental/etc.) breakdown of
  high-confidence hits.
- Publication-ready figures with plot-source data persisted alongside every
  one, following this project's established convention (no on-figure
  titles, no code identifiers in labels, Okabe-Ito colorblind palette,
  300dpi PNG + vector PDF).


In [1]:
# MUST BE FIRST CELL — sets thread-count env vars before numpy/sklearn/
# xgboost/lightgbm are imported (BLAS/OMP thread counts only take effect if
# set before those libraries initialize).
import os
import multiprocessing

HPC_MODE = True  # this notebook runs on HPC, not Colab — set False only if
                  # actually running in Colab (matches notebooks 01/02/03's toggle)

if HPC_MODE:
    N_CORES = int(os.environ.get('NCPUS') or os.environ.get('PBS_NP') or
                  os.environ.get('PBS_NCPUS') or os.environ.get('SLURM_CPUS_PER_TASK') or
                  multiprocessing.cpu_count())
    # NCPUS/PBS_NP/PBS_NCPUS/SLURM_CPUS_PER_TASK reflect what the scheduler
    # actually allocated to THIS job on a shared node — multiprocessing.cpu_count()
    # reads the node's total core count, which oversubscribes badly if the job
    # only got a slice of the node.
    import matplotlib
    matplotlib.use('Agg')
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

# Keep math libraries single-threaded on HPC (matches notebook 3) so a loaded
# model's own predict parallelism never oversubscribes the allocated cores on a
# shared PBS node — screening is a serial loop, no outer joblib parallelism to
# feed, so wide BLAS threads would only risk oversubscription for little gain.
for var in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
            'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[var] = '1' if HPC_MODE else str(N_CORES)

os.environ['OMP_NESTED'] = 'FALSE'
os.environ['MKL_DYNAMIC'] = 'FALSE'

ENV = "HPC" if HPC_MODE else "Colab"
print(f"Environment: {ENV} | Detected cores: {N_CORES} | BLAS/OpenMP threads: {os.environ['OMP_NUM_THREADS']}")


Environment: HPC | Detected cores: 24 | BLAS/OpenMP threads: 1


In [2]:
!pip install rdkit shap crepes venn-abers openpyxl -q


DEPRECATION: Loading egg at /home/smtambo/bioenv/lib/python3.11/site-packages/pdbfixer-1.11.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [3]:
import pandas as pd
import numpy as np
import json
import hashlib
import datetime
import platform
import subprocess
import warnings
from pathlib import Path
from collections import defaultdict

import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import joblib
import venn_abers  # required to unpickle notebook 3's VennAbers calibrator objects
from xml.etree import ElementTree as ET
from crepes import WrapClassifier, WrapRegressor

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.rdBase import BlockLogs
from tqdm import tqdm

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

print('Imports complete.')


Imports complete.


In [4]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path("./")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")

for subdir in ["data/processed", "data/external/drugbank", "ml/models", "ml/results/drugbank_screening", "logs"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

processed_path = PROJECT_DIR / "data" / "processed"
models_path    = PROJECT_DIR / "ml" / "models"
results_path   = PROJECT_DIR / "ml" / "results"
screen_path    = results_path / "drugbank_screening"
logs_path      = PROJECT_DIR / "logs"

print(f"Project directory: {PROJECT_DIR}")


Project directory: .


In [5]:
# =============================================================================
# GLOBAL CONFIGURATION — matches notebook 3 exactly (target/pool/representation
# keys, algorithm labels, Morgan parameters, descriptor columns). No screening
# logic below hardcodes a target-specific value.
# =============================================================================

RANDOM_STATE = 42

TARGETS = ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
POOLS = ['ki', 'ki_ic50', 'full']
POOL_LABELS = {'ki': 'Ki', 'ki_ic50': 'Ki + IC50', 'full': 'Ki + IC50 + EC50'}

FEATURE_REPS = ['morgan', 'descriptors', 'combined']
PRIMARY_REPRESENTATION = 'combined'  # the deployed/reported representation —
                                      # matches notebook 3's deployment tuning
                                      # and SHAP/Y-randomization analyses.

ALGORITHMS = ['rf', 'xgb', 'lgb']
ALGO_LABELS = {'rf': 'Random Forest', 'xgb': 'XGBoost', 'lgb': 'LightGBM'}

# Must match notebook 3 exactly — verified against its MORGAN_RADIUS/MORGAN_NBITS.
MORGAN_RADIUS = 2
MORGAN_NBITS = 2048

# Must match notebook 02 Step 7 / notebook 3 DESCRIPTOR_COLS exactly (same
# RDKit calls, same column order) — these are what every deployed model's
# 'descriptors'/'combined' feature vector was trained on.
DESCRIPTOR_COLS = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds',
                   'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']

HIGH_CONFIDENCE_THRESHOLD = 0.9  # calibrated-probability cutoff for the ranked shortlist
RECOMPUTE_SCREENING_CHECKPOINTS = True  # screening is fast; avoids stale SDF-only metadata caches

# DrugBank's raw structure library is license-restricted and not redistributed
# with this project — set this to wherever you've placed your own copy.
DRUGBANK_SDF_PATH = PROJECT_DIR / 'data' / 'external' / 'drugbank' / 'drugbank.sdf'

# Optional: DrugBank's FULL XML database export (not the SDF) has ATC codes,
# mechanism-of-action, indication, and target/action fields the SDF does not.
# Leave as None to skip — the notebook still runs, those columns are simply
# 'Unknown'. Set the path if you have the XML export and want the richer join.
# Set to your DrugBank full-database XML export if you have one (a separate,
# larger download from the structures SDF, same academic license tier).
# The notebook does NOT parse the raw XML directly -- that's a one-time,
# ~1.5GB streaming-parse preprocessing step (scripts/parse_drugbank_full_xml.py),
# cached to drugbank_full_metadata.csv alongside it. Re-parsing 1.5GB of XML on
# every notebook run would be wasteful; run that script once after placing the
# XML, then this notebook just loads its cached CSV output.
DRUGBANK_FULL_XML_PATH = PROJECT_DIR / 'data' / 'external' / 'drugbank' / 'drugbank_full_database.xml'
DRUGBANK_FULL_METADATA_CSV = DRUGBANK_FULL_XML_PATH.parent / 'drugbank_full_metadata.csv'

# Composite ranking weights (Step 5b) — documented, not a black-box score.
# Each component is min-max normalised to [0, 1] within its (target, pool)
# group before combining, so the weights are directly interpretable as
# relative importance.
PRIORITY_WEIGHTS = {
    'calibrated_proba': 0.40,
    'ad_percentile': 0.25,        # higher percentile = closer to training data (more trustworthy)
    'conformal_confidence': 0.20,  # smaller conformal set / narrower interval = more confident
    'novelty': 0.15,               # not already in training = more novel as a repurposing candidate
}
assert abs(sum(PRIORITY_WEIGHTS.values()) - 1.0) < 1e-9, 'PRIORITY_WEIGHTS must sum to 1.0'

TOP_N_PUBLICATION_TABLE = 20  # deduplicated (one-per-scaffold-cluster) top-N for the publication table

TARGET_COLORS = dict(zip(TARGETS, ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']))

assert set(FEATURE_REPS) == {'morgan', 'descriptors', 'combined'}
assert set(ALGORITHMS) == {'rf', 'xgb', 'lgb'}
print(f'Targets                : {TARGETS}')
print(f'Pools                  : {POOLS}')
print(f'Feature representations: {FEATURE_REPS} (primary: {PRIMARY_REPRESENTATION})')
print(f'DrugBank SDF path      : {DRUGBANK_SDF_PATH}')
print(f'DrugBank SDF exists    : {DRUGBANK_SDF_PATH.exists()}')


Targets                : ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
Pools                  : ['ki', 'ki_ic50', 'full']
Feature representations: ['morgan', 'descriptors', 'combined'] (primary: combined)
DrugBank SDF path      : data/external/drugbank/drugbank.sdf
DrugBank SDF exists    : True


In [6]:
def get_combo_paths(target, pool):
    combo_key = f'{target}_{pool}'
    return {
        'combo_key': combo_key,
        'cleaned_data_path': processed_path / f'cleaned_data_{target}_{pool}.csv',
        'models_dir': models_path / combo_key,
    }


def _hash_file(path, chunk_size=1 << 20):
    '''SHA-256 of a file, chunked so large CSVs don't blow up memory.'''
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def _capture_package_versions():
    packages = ['rdkit', 'numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm',
                'shap', 'joblib', 'crepes', 'scipy', 'matplotlib']
    versions = {}
    for pkg in packages:
        try:
            mod = __import__(pkg)
            versions[pkg] = getattr(mod, '__version__', 'unknown')
        except ImportError:
            pass
    return versions


def write_manifest(manifest_path, config_summary, outputs, inputs=None):
    git_hash = None
    try:
        git_hash = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], stderr=subprocess.DEVNULL, cwd=str(PROJECT_DIR)
        ).decode().strip()
    except Exception:
        pass

    def _hash_registry(registry):
        hashed = {}
        for name, meta in registry.items():
            meta = dict(meta)
            p = Path(meta.get('path', ''))
            if p.exists() and p.is_file():
                try:
                    meta['sha256'] = _hash_file(p)
                except Exception:
                    pass
            hashed[name] = meta
        return hashed

    manifest = {
        'timestamp': datetime.datetime.now().isoformat(),
        'python_version': platform.python_version(),
        'git_commit': git_hash,
        'config': config_summary,
        'inputs': _hash_registry(inputs) if inputs else {},
        'outputs': _hash_registry(outputs),
        'package_versions': _capture_package_versions(),
    }
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2, default=str)
    print(f'Manifest saved: {manifest_path} ({len(manifest["inputs"])} inputs, {len(manifest["outputs"])} outputs hashed)')
    return manifest


all_outputs = {}

def _register(name, path_obj, n_rows=None):
    entry = {'path': str(path_obj)}
    if n_rows is not None:
        entry['n_rows'] = n_rows
    all_outputs[name] = entry


print('Path/manifest helpers defined.')


Path/manifest helpers defined.


In [7]:
# =============================================================================
# PUBLICATION FIGURE STYLE — identical convention to notebooks 02/03: no
# on-figure titles, no code identifiers in labels/legends, American
# spelling, 300dpi PNG + vector PDF, panel letters via panel_label().
# =============================================================================
mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig_dir = screen_path / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

def save_fig(fig, stem):
    for ext in ('png', 'pdf'):
        fig.savefig(fig_dir / f'{stem}.{ext}')
    plt.close(fig)
    _register(f'{stem}.png', fig_dir / f'{stem}.png')
    _register(f'{stem}.pdf', fig_dir / f'{stem}.pdf')

def panel_label(ax, letter):
    '''Bold panel letter (A/B/C...), top-left — NOT a descriptive title.'''
    ax.set_title(letter, loc='left', fontweight='bold', fontsize=12)

print('Figure style configured.')


Figure style configured.


## Step 0: Artifact Compatibility Check

Before touching DrugBank, verify this notebook's feature definitions
(Morgan radius/bits, descriptor columns) match what notebook 3 actually
used to train the models being loaded — read from notebook 3's own
manifest, not assumed. A silent mismatch here (e.g. notebook 3 changes
`MORGAN_RADIUS` later without notebook 4 being updated) would corrupt
every prediction without ever raising an error downstream, since a
2048-bit fingerprint from radius 2 and radius 3 have the same shape but
different meaning. Fails loudly, before any screening runs, rather than
producing silently-wrong predictions.

In [8]:
manifest_03_path = results_path / 'manifest_03_ml_benchmark.json'
assert manifest_03_path.exists(), f'notebook 3 manifest not found: {manifest_03_path} — run notebook 3 first.'
with open(manifest_03_path) as f:
    manifest_03 = json.load(f)
nb3_config = manifest_03.get('config', {})

compat_errors = []
if 'morgan_radius' in nb3_config and nb3_config['morgan_radius'] != MORGAN_RADIUS:
    compat_errors.append(f"MORGAN_RADIUS mismatch: notebook 4 has {MORGAN_RADIUS}, "
                          f"notebook 3 manifest records {nb3_config['morgan_radius']}")
if 'morgan_n_bits' in nb3_config and nb3_config['morgan_n_bits'] != MORGAN_NBITS:
    compat_errors.append(f"MORGAN_NBITS mismatch: notebook 4 has {MORGAN_NBITS}, "
                          f"notebook 3 manifest records {nb3_config['morgan_n_bits']}")
if 'descriptor_columns' in nb3_config and list(nb3_config['descriptor_columns']) != DESCRIPTOR_COLS:
    compat_errors.append(f"DESCRIPTOR_COLS mismatch: notebook 4 has {DESCRIPTOR_COLS}, "
                          f"notebook 3 manifest records {nb3_config['descriptor_columns']}")

# Per-combination feature_names.json is the ground truth actually used at
# prediction time (checked again, per combination, at screening time in
# screen_one_combination() via an explicit shape assertion) — this cell is
# the fast global pre-flight check; that per-combination check is the final,
# authoritative guard.
if compat_errors:
    raise AssertionError('Artifact compatibility check FAILED:\n  - ' + '\n  - '.join(compat_errors))
print('Artifact compatibility check passed: notebook 4 feature definitions match notebook 3\'s manifest.')
print(f'  Morgan radius/bits : {MORGAN_RADIUS} / {MORGAN_NBITS}')
print(f'  Descriptor columns : {DESCRIPTOR_COLS}')


Artifact compatibility check passed: notebook 4 feature definitions match notebook 3's manifest.
  Morgan radius/bits : 2 / 2048
  Descriptor columns : ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']


## Step 1: Load and Standardize DrugBank

Load the DrugBank SDF, standardize every structure with the SAME pipeline
notebook 02 used for the training data (Cleanup, FragmentParent, Normalizer,
Uncharger, TautomerEnumerator), and compute `global_compound_id` with the
SAME SHA-256-of-`clean_smiles` convention — this is what lets us check, per
target, whether a "hit" was already in that target's training set (a check
).

In [9]:
assert DRUGBANK_SDF_PATH.exists(), (
    f'DrugBank SDF not found at {DRUGBANK_SDF_PATH}. DrugBank\'s structure '
    f'library is license-restricted and not redistributed with this project — '
    f'obtain it independently and place it at this path (or update '
    f'DRUGBANK_SDF_PATH above) before continuing.'
)

def clean_structure(mol):
    '''Same standardisation sequence as notebook 02 Step 1, applied to a
    single already-parsed RDKit mol (DrugBank SDF gives us mols directly,
    no SMILES-parsing step needed first).'''
    try:
        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)
        mol = rdMolStandardize.Normalizer().normalize(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
        canon_mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
        return Chem.MolToSmiles(canon_mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None


# Permissive metadata parsing: SDF property names vary by DrugBank export
# version, and several of these (ATC, MOA, indication, targets) are usually
# only present in the FULL XML export, not the basic structure SDF — grab
# whatever properties exist, leave 'Unknown' for whatever doesn't. The
# notebook runs identically either way; richer metadata just fills in more
# columns if your DrugBank export has them.
SDF_METADATA_PROPS = {
    'drugbank_id': ['DRUGBANK_ID', 'DATABASE_ID'],
    'generic_name': ['GENERIC_NAME', 'NAME'],
    'drug_groups': ['DRUG_GROUPS', 'GROUPS'],
    'atc_codes': ['ATC_CODES', 'ATC-CODE'],
    'indication': ['INDICATION'],
    'mechanism_of_action': ['MECHANISM_OF_ACTION', 'PHARMACOLOGY'],
    'targets': ['TARGET', 'DRUG_TARGET_1_NAME', 'TARGETS'],
}

def _first_available_prop(mol, candidates):
    for prop in candidates:
        if mol.HasProp(prop):
            return mol.GetProp(prop)
    return 'Unknown'

print(f'Loading DrugBank SDF: {DRUGBANK_SDF_PATH}')
suppl = Chem.SDMolSupplier(str(DRUGBANK_SDF_PATH))

drugbank_rows = []
block = BlockLogs()
for mol in tqdm(suppl, desc='Standardising DrugBank structures'):
    if mol is None:
        continue
    row = {field: _first_available_prop(mol, props) for field, props in SDF_METADATA_PROPS.items()}
    clean_smiles = clean_structure(mol)
    if clean_smiles is None:
        continue
    row['clean_smiles'] = clean_smiles
    drugbank_rows.append(row)
del block

_available_fields = [f for f in SDF_METADATA_PROPS
                      if any(r[f] != 'Unknown' for r in drugbank_rows[:min(500, len(drugbank_rows))])]
_missing_fields = [f for f in SDF_METADATA_PROPS if f not in _available_fields]
print(f'Metadata fields found in this SDF: {_available_fields}')
if _missing_fields:
    print(f'Metadata fields NOT in this SDF (will read \'Unknown\'; need the full DrugBank XML export '
          f'for these — set DRUGBANK_FULL_XML_PATH): {_missing_fields}')

drugbank_df = pd.DataFrame(drugbank_rows).drop_duplicates(subset='clean_smiles').reset_index(drop=True)
drugbank_df['global_compound_id'] = drugbank_df['clean_smiles'].apply(
    lambda s: 'CMPD_' + hashlib.sha256(s.encode('utf-8')).hexdigest()[:12])

n_raw = len(list(Chem.SDMolSupplier(str(DRUGBANK_SDF_PATH))))
print(f'DrugBank: {n_raw} raw records -> {len(drugbank_df)} standardised, deduplicated compounds')
print('\nDrug-group breakdown (top 10):')
print(drugbank_df['drug_groups'].value_counts().head(10).to_string())


Loading DrugBank SDF: data/external/drugbank/drugbank.sdf


Standardising DrugBank structures: 100%|██████████| 10041/10041 [04:28<00:00, 37.34it/s]


Metadata fields found in this SDF: ['drugbank_id', 'generic_name', 'drug_groups']
Metadata fields NOT in this SDF (will read 'Unknown'; need the full DrugBank XML export for these — set DRUGBANK_FULL_XML_PATH): ['atc_codes', 'indication', 'mechanism_of_action', 'targets']
DrugBank: 10041 raw records -> 9874 standardised, deduplicated compounds

Drug-group breakdown (top 10):
drug_groups
experimental                     4803
investigational                  2594
approved; investigational         755
approved                          754
approved; vet_approved            106
experimental; illicit              91
experimental; investigational      83
approved; experimental             82
approved; withdrawn                74
vet_approved                       63


In [10]:
# ---- Optional enrichment: DrugBank full-database metadata (indication,
# mechanism-of-action, ATC codes, human protein targets/actions). Fully
# self-contained: if DRUGBANK_FULL_XML_PATH exists and the cached parse
# (DRUGBANK_FULL_METADATA_CSV) doesn't yet, this cell streams the raw XML
# ONCE with iterparse (bounded memory regardless of the ~1.5GB file size --
# do not change this to a full ElementTree.parse()/fromstring() load) and
# caches the result, so re-running this notebook never re-parses the XML.
# If neither the cache nor the raw XML is present, this is a no-op and every
# downstream table keeps 'Unknown' exactly as before -- runs identically
# either way. ----
_DB_NS = '{http://www.drugbank.ca}'

def _drugbank_xml_text(drug_elem, tag):
    el = drug_elem.find(f'{_DB_NS}{tag}')
    if el is None or el.text is None:
        return None
    return ' '.join(el.text.split())

def _drugbank_xml_atc_codes(drug_elem):
    codes = [a.get('code') for a in drug_elem.findall(f'{_DB_NS}atc-codes/{_DB_NS}atc-code') if a.get('code')]
    return '; '.join(codes) if codes else None

def _drugbank_xml_human_targets(drug_elem):
    entries = []
    for target in drug_elem.findall(f'{_DB_NS}targets/{_DB_NS}target'):
        organism = target.find(f'{_DB_NS}organism')
        if organism is None or (organism.text or '').strip() != 'Humans':
            continue
        name_el = target.find(f'{_DB_NS}name')
        name = (name_el.text or '').strip() if name_el is not None else None
        if not name:
            continue
        actions = [(a.text or '').strip() for a in target.findall(f'{_DB_NS}actions/{_DB_NS}action') if (a.text or '').strip()]
        entries.append(f"{name} ({'/'.join(actions)})" if actions else name)
    return '; '.join(entries) if entries else None

def parse_drugbank_full_xml(xml_path):
    '''One-time streaming parse of DrugBank's full-database XML export into a
    flat per-drug metadata table. Clears each <drug> element after reading it
    so memory stays bounded no matter how large the input file is.'''
    rows = []
    n_drugs = 0
    for event, elem in ET.iterparse(str(xml_path), events=('end',)):
        if elem.tag != f'{_DB_NS}drug':
            continue
        id_elem = elem.find(f'{_DB_NS}drugbank-id[@primary="true"]')
        if id_elem is None or not id_elem.text:
            elem.clear()
            continue
        rows.append({
            'drugbank_id': id_elem.text.strip(),
            'indication': _drugbank_xml_text(elem, 'indication'),
            'mechanism_of_action': _drugbank_xml_text(elem, 'mechanism-of-action'),
            'atc_codes_full': _drugbank_xml_atc_codes(elem),
            'human_targets_full': _drugbank_xml_human_targets(elem),
        })
        n_drugs += 1
        if n_drugs % 5000 == 0:
            print(f'  parsed {n_drugs} DrugBank XML entries...')
        elem.clear()
    return pd.DataFrame(rows)

if not DRUGBANK_FULL_METADATA_CSV.exists() and DRUGBANK_FULL_XML_PATH.exists():
    print(f'Parsing DrugBank full XML (one-time, cached after this): {DRUGBANK_FULL_XML_PATH}')
    full_meta_df = parse_drugbank_full_xml(DRUGBANK_FULL_XML_PATH)
    full_meta_df.to_csv(DRUGBANK_FULL_METADATA_CSV, index=False)
    print(f'Parsed {len(full_meta_df)} DrugBank entries -> cached at {DRUGBANK_FULL_METADATA_CSV}')

if DRUGBANK_FULL_METADATA_CSV.exists():
    full_meta_df = pd.read_csv(DRUGBANK_FULL_METADATA_CSV)
    drugbank_df = drugbank_df.merge(full_meta_df, on='drugbank_id', how='left', suffixes=('', '_full'))

    drugbank_df['indication'] = drugbank_df['indication_full'].where(
        drugbank_df['indication_full'].notna(), drugbank_df['indication'])
    drugbank_df['mechanism_of_action'] = drugbank_df['mechanism_of_action_full'].where(
        drugbank_df['mechanism_of_action_full'].notna(), drugbank_df['mechanism_of_action'])
    drugbank_df['atc_codes'] = drugbank_df['atc_codes_full'].where(
        drugbank_df['atc_codes_full'].notna(), drugbank_df['atc_codes'])
    drugbank_df['targets'] = drugbank_df['human_targets_full'].where(
        drugbank_df['human_targets_full'].notna(), drugbank_df['targets'])
    drugbank_df = drugbank_df.drop(columns=[
        'indication_full', 'mechanism_of_action_full', 'atc_codes_full', 'human_targets_full',
    ])

    n_ind = int((drugbank_df['indication'] != 'Unknown').sum())
    n_moa = int((drugbank_df['mechanism_of_action'] != 'Unknown').sum())
    n_tgt = int((drugbank_df['targets'] != 'Unknown').sum())
    print(f'DrugBank full-metadata enrichment applied:')
    print(f'  indication known for           {n_ind} / {len(drugbank_df)} compounds')
    print(f'  mechanism_of_action known for  {n_moa} / {len(drugbank_df)} compounds')
    print(f'  human targets known for        {n_tgt} / {len(drugbank_df)} compounds')
else:
    print(f'No DrugBank full XML or cached metadata found at {DRUGBANK_FULL_XML_PATH} -- '
          f"indication/mechanism_of_action/targets remain 'Unknown'.")


DrugBank full-metadata enrichment applied:
  indication known for           2421 / 9874 compounds
  mechanism_of_action known for  2270 / 9874 compounds
  human targets known for        5708 / 9874 compounds


In [11]:
# ---- Save the standardised DrugBank library (compute once, reuse for every
# target/pool/representation below) ----
drugbank_library_path = screen_path / 'drugbank_library_standardised.csv'
drugbank_df.to_csv(drugbank_library_path, index=False)
_register('drugbank_library_standardised.csv', drugbank_library_path, len(drugbank_df))
print(f'Standardised DrugBank library saved: {drugbank_library_path}')


Standardised DrugBank library saved: ml/results/drugbank_screening/drugbank_library_standardised.csv


## Step 2: Feature Generation (Morgan, descriptors, combined)

Generated exactly once for the whole DrugBank library, in the SAME column
order every deployed model expects (verified against each combination's
saved `feature_names.json` at prediction time, not assumed).

In [12]:
def smiles_to_mols(smiles_list):
    mols = []
    with BlockLogs():
        for smi in smiles_list:
            mols.append(Chem.MolFromSmiles(smi) if isinstance(smi, str) else None)
    return mols


def compute_morgan_matrix(mols, radius=MORGAN_RADIUS, n_bits=MORGAN_NBITS):
    fps = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, mol in enumerate(mols):
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
        arr = np.zeros(n_bits, dtype=np.uint8)
        for on_bit in fp.GetOnBits():
            arr[on_bit] = 1
        fps[i] = arr
    return fps


def compute_descriptor_matrix(mols):
    '''Exact same RDKit calls, same order, as notebook 02 Step 7 / notebook 3
    DESCRIPTOR_COLS — this is what every deployed 'descriptors'/'combined'
    model was trained on.'''
    rows = []
    with BlockLogs():
        for mol in mols:
            if mol is None:
                rows.append([np.nan] * len(DESCRIPTOR_COLS))
                continue
            rows.append([
                Descriptors.MolWt(mol), Crippen.MolLogP(mol), Descriptors.TPSA(mol),
                Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol),
                Descriptors.NumRotatableBonds(mol), Descriptors.HeavyAtomCount(mol),
                Descriptors.RingCount(mol), Descriptors.NumAromaticRings(mol),
                Descriptors.FractionCSP3(mol),
            ])
    return np.array(rows, dtype=float)


drugbank_mols = smiles_to_mols(drugbank_df['clean_smiles'].tolist())
n_invalid = sum(m is None for m in drugbank_mols)
if n_invalid > 0:
    print(f'WARNING: {n_invalid} DrugBank compounds failed to re-parse — zero-vector features for those rows')

X_morgan = compute_morgan_matrix(drugbank_mols)
X_desc = compute_descriptor_matrix(drugbank_mols)
X_desc = np.nan_to_num(X_desc, nan=0.0, posinf=0.0, neginf=0.0)
X_combined = np.hstack([X_desc, X_morgan])

X_by_rep = {'morgan': X_morgan, 'descriptors': X_desc, 'combined': X_combined}
print(f'DrugBank feature matrices: morgan={X_morgan.shape} descriptors={X_desc.shape} combined={X_combined.shape}')

np.save(screen_path / 'drugbank_features_morgan.npy', X_morgan)
np.save(screen_path / 'drugbank_features_descriptors.npy', X_desc)
np.save(screen_path / 'drugbank_features_combined.npy', X_combined)
_register('drugbank_features_morgan.npy', screen_path / 'drugbank_features_morgan.npy', X_morgan.shape[0])
_register('drugbank_features_descriptors.npy', screen_path / 'drugbank_features_descriptors.npy', X_desc.shape[0])
_register('drugbank_features_combined.npy', screen_path / 'drugbank_features_combined.npy', X_combined.shape[0])


DrugBank feature matrices: morgan=(9874, 2048) descriptors=(9874, 10) combined=(9874, 2058)


## Step 3: Screen — Apply Every Deployed Model

For each (target, pool, representation): load the deployed classifier and
regressor, the Platt calibrator, both conformal wrappers, and the frozen
applicability-domain reference — all already fit by notebook 3, nothing
refit here — and score the entire DrugBank library. The algorithm used per
(target, pool) is read from notebook 3's `best_algorithm_by_combination.csv`
(the leak-free, inner-CV-selected choice), never re-derived here.

In [13]:
best_algo_path = results_path / 'best_algorithm_by_combination.csv'
assert best_algo_path.exists(), f'notebook 3 output not found: {best_algo_path} — run notebook 3 first.'
best_algorithm_df = pd.read_csv(best_algo_path)
ALGO_LABEL_TO_KEY = {v: k for k, v in ALGO_LABELS.items()}

def get_best_algo_key(target, pool, task):
    row = best_algorithm_df[
        (best_algorithm_df['target'] == target) & (best_algorithm_df['activity_pool'] == pool) &
        (best_algorithm_df['task'] == task)
    ]
    if len(row) == 0:
        return None
    return ALGO_LABEL_TO_KEY[row.iloc[0]['best_algorithm']]

print(f'Loaded best-algorithm determinations for {len(best_algorithm_df)} (target, pool, task) combinations.')

# Integrity gate: the selection table must describe the same deployed models
# notebook 3 selected from inner-CV tuning histories. This prevents downstream
# screening from using a stale best_algorithm_by_combination.csv.
def assert_deployed_selection_integrity(scope_targets=TARGETS, scope_pools=POOLS,
                                        scope_tasks=('classification', 'regression'),
                                        representation=PRIMARY_REPRESENTATION):
    label_to_stem = {'Random Forest': 'rf', 'XGBoost': 'xgb', 'LightGBM': 'lgb'}
    stem_to_label = {v: k for k, v in label_to_stem.items()}
    failures = []
    checked = 0
    for target in scope_targets:
        for pool in scope_pools:
            history_path = results_path / f'{target}_{pool}' / 'optuna_tuning_history.json'
            history = json.load(open(history_path)) if history_path.exists() else {}
            for task in scope_tasks:
                row = best_algorithm_df[
                    (best_algorithm_df['target'] == target) &
                    (best_algorithm_df['activity_pool'] == pool) &
                    (best_algorithm_df['task'] == task)
                ]
                if len(row) != 1:
                    failures.append(f'{target}/{pool}/{task}: expected one selection row, found {len(row)}')
                    continue
                recorded = str(row.iloc[0]['best_algorithm'])
                values = {}
                for key, entry in history.items():
                    if key.endswith(task) and isinstance(entry, dict) and 'best_value' in entry:
                        stem = key.split('_')[0]
                        if stem in stem_to_label:
                            values[stem_to_label[stem]] = float(entry['best_value'])
                if values:
                    recomputed = max(values, key=values.get)
                    if recorded != recomputed:
                        failures.append(f'{target}/{pool}/{task}: selection table says {recorded}, tuning history says {recomputed}')
                stem = label_to_stem.get(recorded)
                if stem is None:
                    failures.append(f'{target}/{pool}/{task}: unknown algorithm label {recorded}')
                    continue
                artifact = f'{stem}_clf.joblib' if task == 'classification' else f'{stem}_reg.joblib'
                model_file = models_path / f'{target}_{pool}' / representation / artifact
                if not model_file.exists():
                    failures.append(f'{target}/{pool}/{task}: deployed artifact missing: {model_file}')
                checked += 1
    if failures:
        raise RuntimeError('Deployed-model selection integrity check failed:\n  - ' + '\n  - '.join(failures))
    print(f'Deployed-model selection integrity check passed for {checked} target/pool/task entries.')

assert_deployed_selection_integrity()



Loaded best-algorithm determinations for 30 (target, pool, task) combinations.
Deployed-model selection integrity check passed for 30 target/pool/task entries.


In [14]:
def apply_applicability_domain(ad_ref, X_query):
    X_scaled = ad_ref['scaler'].transform(X_query.astype(float))
    dists, _ = ad_ref['knn'].kneighbors(X_scaled)
    knn_mean = dists[:, :ad_ref['k']].mean(axis=1)
    within = knn_mean <= ad_ref['threshold']
    return knn_mean, within


def screen_one_combination(target, pool, representation):
    '''Apply target/pool/representation's deployed, calibrated, leak-free
    model to the full DrugBank library. Returns a per-compound DataFrame, or
    None if the required artifacts are missing (combination not yet trained).'''
    paths = get_combo_paths(target, pool)
    rep_dir = paths['models_dir'] / representation
    clf_key = get_best_algo_key(target, pool, 'classification')
    reg_key = get_best_algo_key(target, pool, 'regression')
    if clf_key is None:
        return None

    clf_path = rep_dir / f'{clf_key}_clf.joblib'
    ad_path = rep_dir / 'ad_reference.pkl'
    cal_path = rep_dir / f'{clf_key}_clf_calibrator.pkl'
    conformal_clf_path = rep_dir / f'{clf_key}_clf_conformal.pkl'
    feature_names_path = rep_dir / 'feature_names.json'
    if not (clf_path.exists() and ad_path.exists() and feature_names_path.exists()):
        print(f'  SKIP {target}/{pool}/{representation}: required model artifacts not found in {rep_dir}')
        return None

    with open(feature_names_path) as f:
        feature_names = json.load(f)
    X = X_by_rep[representation]
    assert X.shape[1] == len(feature_names), (
        f'{target}/{pool}/{representation}: DrugBank feature width {X.shape[1]} != '
        f'trained feature width {len(feature_names)} — feature definitions have drifted.'
    )

    clf = joblib.load(clf_path)
    ad_ref = joblib.load(ad_path)
    # Probability of the ACTIVE class (label==1), located via clf.classes_
    # rather than assuming it is column 1.
    _active_proba_col = list(clf.classes_).index(1) if 1 in list(clf.classes_) else 1
    proba_raw = clf.predict_proba(X)[:, _active_proba_col]

    proba_cal = proba_raw
    cal_interval_width = np.full(len(X), np.nan)
    if cal_path.exists():
        # Venn-ABERS calibrator (VennAbers class) -- takes the full 2-column
        # predict_proba output directly (no logit transform) and returns
        # (p_prime, p0_p1); p0_p1's width is a real per-compound calibration-
        # uncertainty signal, persisted below rather than discarded.
        venn_abers_calibrator = joblib.load(cal_path)
        p_prime, p0_p1 = venn_abers_calibrator.predict_proba(clf.predict_proba(X))
        proba_cal = p_prime[:, 1]
        cal_interval_width = p0_p1[:, 1] - p0_p1[:, 0]

    # crepes WrapClassifier.predict_set returns a boolean (n, n_classes) matrix
    # whose COLUMN ORDER follows clf.classes_ — do NOT assume column 0 is the
    # inactive class; look up which column is the active (label==1) class via
    # clf.classes_ so this stays correct even if a model's class order differs.
    # set_size==1 means only one class survived (a confident call); set_size==2
    # means both classes are in the set (the model is uncertain at this
    # confidence level, whatever raw probability it output).
    conformal_set_includes_inactive = np.full(len(X), np.nan)
    conformal_set_includes_active = np.full(len(X), np.nan)
    conformal_set_size = np.full(len(X), np.nan)
    if conformal_clf_path.exists():
        wrapped_clf = joblib.load(conformal_clf_path)
        # labels=False: binary indicator array, matching the column-order lookup
        # via clf.classes_ below (default labels=True returns label-name lists,
        # which breaks the [:, col] indexing).
        pred_set = wrapped_clf.predict_set(X, confidence=0.9, labels=False)
        classes = list(clf.classes_)
        active_col = classes.index(1) if 1 in classes else 1
        inactive_col = classes.index(0) if 0 in classes else 0
        conformal_set_includes_inactive = pred_set[:, inactive_col].astype(bool)
        conformal_set_includes_active = pred_set[:, active_col].astype(bool)
        conformal_set_size = pred_set.sum(axis=1).astype(int)

    ad_dist, within_ad = apply_applicability_domain(ad_ref, X)

    reg_pred = np.full(len(X), np.nan)
    reg_lo90 = reg_hi90 = np.full(len(X), np.nan)
    if reg_key is not None:
        reg_path = rep_dir / f'{reg_key}_reg.joblib'
        conformal_reg_path = rep_dir / f'{reg_key}_reg_conformal.pkl'
        if reg_path.exists():
            reg = joblib.load(reg_path)
            reg_pred = reg.predict(X)
        if conformal_reg_path.exists():
            wrapped_reg = joblib.load(conformal_reg_path)
            interval = wrapped_reg.predict_int(X, confidence=0.9)
            reg_lo90, reg_hi90 = interval[:, 0], interval[:, 1]

    _meta_cols = ['drugbank_id', 'generic_name', 'drug_groups', 'atc_codes', 'indication',
                  'mechanism_of_action', 'targets', 'clean_smiles', 'global_compound_id']
    out = drugbank_df[_meta_cols].copy()
    out['target'] = target
    out['activity_pool'] = pool
    out['feature_representation'] = representation
    out['classification_algorithm'] = ALGO_LABELS[clf_key]
    out['regression_algorithm'] = ALGO_LABELS[reg_key] if reg_key is not None else None
    # 'algorithm' kept as an alias of the classifier algorithm for backward
    # compatibility with downstream cells; classifier and regressor can differ,
    # so both are now recorded explicitly above.
    out['algorithm'] = ALGO_LABELS[clf_key]
    out['predicted_proba_raw'] = proba_raw
    out['predicted_proba_calibrated'] = proba_cal
    out['calibration_interval_width'] = cal_interval_width
    out['conformal_set_includes_inactive_90pct'] = conformal_set_includes_inactive
    out['conformal_set_includes_active_90pct'] = conformal_set_includes_active
    out['conformal_set_size_90pct'] = conformal_set_size
    out['predicted_pactivity'] = reg_pred
    out['pactivity_interval_lo_90pct'] = reg_lo90
    out['pactivity_interval_hi_90pct'] = reg_hi90
    out['within_applicability_domain'] = within_ad
    out['applicability_domain_distance'] = ad_dist
    return out


In [15]:
# =============================================================================
# MASTER SCREENING LOOP — every (target x pool x representation) already
# trained by notebook 3. Checkpointed per combination.
# =============================================================================
screening_frames = []

for target in TARGETS:
    for pool in POOLS:
        for representation in FEATURE_REPS:
            ckpt_path = screen_path / f'screening_{target}_{pool}_{representation}.csv'
            if ckpt_path.exists() and not RECOMPUTE_SCREENING_CHECKPOINTS:
                print(f'  ⏭️  {target}/{pool}/{representation}: checkpoint found, loading cached results')
                result_df = pd.read_csv(ckpt_path)
            else:
                result_df = screen_one_combination(target, pool, representation)
                if result_df is None:
                    continue
                result_df.to_csv(ckpt_path, index=False)
                _register(ckpt_path.name, ckpt_path, len(result_df))
            screening_frames.append(result_df)
        print(f'{target}/{pool}: screening complete for all representations')

screening_all_df = pd.concat(screening_frames, ignore_index=True)
screening_all_path = screen_path / 'screening_results_all.csv'
screening_all_df.to_csv(screening_all_path, index=False)
_register('screening_results_all.csv', screening_all_path, len(screening_all_df))
print(f'\nAll screening results saved: {screening_all_path} ({len(screening_all_df)} rows)')


OMP: Info #268: OMP_NESTED variable deprecated, please use OMP_MAX_ACTIVE_LEVELS instead.


drd2/ki: screening complete for all representations
drd2/ki_ic50: screening complete for all representations
drd2/full: screening complete for all representations
cb2/ki: screening complete for all representations
cb2/ki_ic50: screening complete for all representations
cb2/full: screening complete for all representations
adora2a/ki: screening complete for all representations
adora2a/ki_ic50: screening complete for all representations
adora2a/full: screening complete for all representations
oprm1/ki: screening complete for all representations
oprm1/ki_ic50: screening complete for all representations
oprm1/full: screening complete for all representations
ccr5/ki: screening complete for all representations
ccr5/ki_ic50: screening complete for all representations
ccr5/full: screening complete for all representations

All screening results saved: ml/results/drugbank_screening/screening_results_all.csv (444330 rows)


## Step 4: Already-in-Training Cross-Reference

Flags any DrugBank hit whose `global_compound_id` already appears in that
target's training data (notebook 2's cleaned dataset)  — the earlier implementation
project never checked this, so a "novel repurposing candidate" could
silently have been a compound the model was trained on. Computed once per
target/pool, joined onto every representation's screening result.

In [16]:
training_id_sets = {}
for target in TARGETS:
    for pool in POOLS:
        paths = get_combo_paths(target, pool)
        if not paths['cleaned_data_path'].exists():
            continue
        cleaned = pd.read_csv(paths['cleaned_data_path'], usecols=['clean_smiles'])
        ids = cleaned['clean_smiles'].apply(
            lambda s: 'CMPD_' + hashlib.sha256(str(s).encode('utf-8')).hexdigest()[:12])
        training_id_sets[(target, pool)] = set(ids)

screening_all_df['already_in_training_set'] = screening_all_df.apply(
    lambda row: row['global_compound_id'] in training_id_sets.get((row['target'], row['activity_pool']), set()),
    axis=1,
)
screening_all_df.to_csv(screening_all_path, index=False)
_register('screening_results_all.csv', screening_all_path, len(screening_all_df))

n_flagged = int(screening_all_df['already_in_training_set'].sum())
print(f'{n_flagged} / {len(screening_all_df)} screening rows are compounds already present in that '
      f'target\'s training data (flagged, not removed — visible in every downstream table).')


2778 / 444330 screening rows are compounds already present in that target's training data (flagged, not removed — visible in every downstream table).


## Step 5: Primary Deployed-Model Screening, Rediscovery, and Ranked Candidates

Two analytical roles are separated explicitly:

- **Sensitivity screening** - all activity pools and feature representations remain
  available in `screening_results_all.csv` and supporting consensus summaries.
  These outputs assess robustness to endpoint-pool / representation choice and do
  **not** nominate primary prospective candidates.
- **Primary prospective screening** - restricted to `activity_pool == 'full'` and
  `feature_representation == 'combined'`. Therefore each target's DrugBank
  shortlist is generated by the same deployed full-pool classifier used in the
  manuscript's primary model-selection / validation chain.

Within the primary screen, two distinct outputs are produced:

- **Known rediscovery** - high-confidence hits already in that target's full-pool
  training data; a sanity check, not a novel finding.
- **Novel candidates** - high-confidence hits not already in that target's
  full-pool training data; the prospective shortlist used for downstream docking.

Both are ranked transparently by calibrated probability and by the documented
composite priority score.


In [17]:
# Preserve the combined-representation results across ALL endpoint pools as a
# sensitivity layer. These rows are useful for robustness analyses, but they
# MUST NOT nominate compounds to the primary DrugBank -> docking funnel.
sensitivity_combined_df = screening_all_df[
    screening_all_df['feature_representation'] == PRIMARY_REPRESENTATION
].copy()

# PRIMARY prospective DrugBank screen (Option A): one deployed model per target.
# Restrict candidate nomination to the FULL activity pool + COMBINED representation.
# Notebook 3's best_algorithm_by_combination.csv supplies the deployment-selected
# classifier/regressor for each target/full combination, so this preserves a single
# provenance chain: model selection -> external validation -> DrugBank -> docking.
primary_df = sensitivity_combined_df[
    sensitivity_combined_df['activity_pool'] == 'full'
].copy()

assert set(primary_df['activity_pool'].dropna().unique()) <= {'full'},     'Primary DrugBank screen must contain full-pool rows only.'
assert set(primary_df['feature_representation'].dropna().unique()) <= {PRIMARY_REPRESENTATION},     'Primary DrugBank screen must contain combined-representation rows only.'

print(f'All-pool combined-representation sensitivity rows: {len(sensitivity_combined_df):,}')
print(f'PRIMARY deployed full/combined screening rows: {len(primary_df):,}')
print('Primary classifier/regressor provenance by target:')
_algo_check = primary_df[['target', 'activity_pool',
                          'classification_algorithm', 'regression_algorithm']].drop_duplicates()
print(_algo_check.to_string(index=False))



All-pool combined-representation sensitivity rows: 148,110
PRIMARY deployed full/combined screening rows: 49,370
Primary classifier/regressor provenance by target:
 target activity_pool classification_algorithm regression_algorithm
   drd2          full                  XGBoost              XGBoost
    cb2          full                 LightGBM              XGBoost
adora2a          full                  XGBoost              XGBoost
  oprm1          full                  XGBoost              XGBoost
   ccr5          full            Random Forest              XGBoost


In [18]:
# =============================================================================
# COMPOSITE PRIORITY SCORE -- documented, transparent, not a black box.
# Each component min-max normalised to [0, 1] WITHIN each target's primary full-pool group
# before combining, so the weights (PRIORITY_WEIGHTS, config cell) are directly
# interpretable as relative importance, and no single target's scale dominates.
# =============================================================================

def _minmax_norm(s):
    lo, hi = s.min(), s.max()
    if hi - lo < 1e-12:
        return pd.Series(0.5, index=s.index)  # no spread in this group -> neutral
    return (s - lo) / (hi - lo)


def compute_priority_score(df):
    df = df.copy()
    parts = []
    for (target, pool), grp_idx in df.groupby(['target', 'activity_pool']).groups.items():
        grp = df.loc[grp_idx]

        calibrated_component = grp['predicted_proba_calibrated']

        # AD percentile: SMALLER ad distance = closer to training data = MORE
        # trustworthy, so invert the rank (1 - normalised distance).
        ad_filled = grp['applicability_domain_distance'].fillna(grp['applicability_domain_distance'].max())
        ad_component = 1.0 - _minmax_norm(ad_filled)

        # Conformal confidence: combine classification set size (1=confident,
        # 2=uncertain) and, where available, regression interval width --
        # smaller/narrower = more confident. Average whichever are available.
        conf_parts = []
        if grp['conformal_set_size_90pct'].notna().any():
            conf_parts.append(1.0 - _minmax_norm(grp['conformal_set_size_90pct'].fillna(2)))
        interval_width = grp['pactivity_interval_hi_90pct'] - grp['pactivity_interval_lo_90pct']
        if interval_width.notna().any():
            conf_parts.append(1.0 - _minmax_norm(interval_width.fillna(interval_width.max())))
        if conf_parts:
            conformal_component = pd.concat(conf_parts, axis=1).mean(axis=1)
        else:
            conformal_component = pd.Series(0.5, index=grp.index)

        novelty_component = (~grp['already_in_training_set']).astype(float)

        score = (
            PRIORITY_WEIGHTS['calibrated_proba'] * calibrated_component +
            PRIORITY_WEIGHTS['ad_percentile'] * ad_component +
            PRIORITY_WEIGHTS['conformal_confidence'] * conformal_component +
            PRIORITY_WEIGHTS['novelty'] * novelty_component
        )
        part_df = pd.DataFrame({
            'priority_score': score,
            'priority_calibrated_component': calibrated_component,
            'priority_ad_component': ad_component,
            'priority_conformal_component': conformal_component,
            'priority_novelty_component': novelty_component,
        }, index=grp.index)
        parts.append(part_df)

    return df.join(pd.concat(parts))


primary_df = compute_priority_score(primary_df)
print('Composite priority score computed for all primary-representation screening rows.')
print(f'Weights: {PRIORITY_WEIGHTS}')


Composite priority score computed for all primary-representation screening rows.
Weights: {'calibrated_proba': 0.4, 'ad_percentile': 0.25, 'conformal_confidence': 0.2, 'novelty': 0.15}


In [19]:
# ---- Known rediscovery: high-confidence hits already in training (sanity check) ----
rediscovery_df = primary_df[
    (primary_df['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD) &
    (primary_df['already_in_training_set'])
].sort_values('predicted_proba_calibrated', ascending=False)

rediscovery_path = screen_path / 'drugbank_known_rediscovery.csv'
rediscovery_df.to_csv(rediscovery_path, index=False)
_register('drugbank_known_rediscovery.csv', rediscovery_path, len(rediscovery_df))
print(f'Known rediscovery (high-confidence, already in training): {len(rediscovery_df)} rows -> {rediscovery_path}')
if len(rediscovery_df):
    print(rediscovery_df.groupby('target').size().to_string())

# ---- Novel candidates: high-confidence hits NOT already in training (the actual shortlist) ----
novel_df = primary_df[
    (primary_df['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD) &
    (~primary_df['already_in_training_set'])
].copy()

# Two rankings of the same novel set: raw calibrated probability (simplest,
# transparent) and composite priority score (accounts for AD/conformal/novelty
# too) -- both saved, neither hidden.
by_probability_df = novel_df.sort_values('predicted_proba_calibrated', ascending=False)
by_probability_path = screen_path / 'drugbank_novel_candidates_ranked_by_probability.csv'
by_probability_df.to_csv(by_probability_path, index=False)
_register('drugbank_novel_candidates_ranked_by_probability.csv', by_probability_path, len(by_probability_df))

ranked_df = novel_df.sort_values('priority_score', ascending=False)
ranked_path = screen_path / 'drugbank_primary_ranked_candidates.csv'
ranked_df.to_csv(ranked_path, index=False)
_register('drugbank_primary_ranked_candidates.csv', ranked_path, len(ranked_df))

print(f'\nNovel candidates (high-confidence, not in training): {len(novel_df)} rows')
print(f'  Ranked by calibrated probability -> {by_probability_path}')
print(f'  Ranked by composite priority score -> {ranked_path}')
if len(novel_df):
    print('\nPer-target novel-candidate counts:')
    print(novel_df.groupby('target').size().to_string())
    print('\nDrug-group breakdown of novel candidates (top 10):')
    print(novel_df['drug_groups'].value_counts().head(10).to_string())
    within_ad_pct = novel_df['within_applicability_domain'].mean() * 100
    print(f'\nWithin applicability domain: {within_ad_pct:.1f}% of novel candidates')

# shortlist_df kept as an alias (= novel_df, priority-ranked) so later cells
# (figures, closing summary) that already reference it keep working.
shortlist_df = ranked_df


Known rediscovery (high-confidence, already in training): 91 rows -> ml/results/drugbank_screening/drugbank_known_rediscovery.csv
target
adora2a     5
cb2        13
ccr5        4
drd2       37
oprm1      32

Novel candidates (high-confidence, not in training): 92 rows
  Ranked by calibrated probability -> ml/results/drugbank_screening/drugbank_novel_candidates_ranked_by_probability.csv
  Ranked by composite priority score -> ml/results/drugbank_screening/drugbank_primary_ranked_candidates.csv

Per-target novel-candidate counts:
target
adora2a     3
cb2         9
drd2       29
oprm1      51

Drug-group breakdown of novel candidates (top 10):
drug_groups
investigational                         25
experimental                            19
illicit                                 11
approved; investigational               10
approved                                 9
experimental; illicit                    8
approved; withdrawn                      2
approved; investigational; withdrawn  

## Step 6: Cross-Target Polypharmacology

DrugBank compounds predicted active above the high-confidence threshold by the
**primary deployed full-pool + combined-representation models** against more than
one of the 5 targets. This is descriptive only and flags candidates whose
repurposing signal may reflect broader GPCR polypharmacology rather than target
selectivity.


In [20]:
hit_mask = primary_df['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD
full_pool_hits = primary_df[hit_mask]

polypharm_records = []
for compound_id, grp in full_pool_hits.groupby('global_compound_id'):
    targets_hit = sorted(grp['target'].unique())
    if len(targets_hit) < 2:
        continue
    polypharm_records.append({
        'global_compound_id': compound_id,
        'drugbank_id': grp['drugbank_id'].iloc[0],
        'generic_name': grp['generic_name'].iloc[0],
        'n_targets_hit': len(targets_hit),
        'targets_hit': '; '.join(t.upper() for t in targets_hit),
        'mean_calibrated_proba': float(grp['predicted_proba_calibrated'].mean()),
    })

polypharm_columns = ['global_compound_id', 'drugbank_id', 'generic_name', 'n_targets_hit', 'targets_hit', 'mean_calibrated_proba']
polypharm_df = (pd.DataFrame(polypharm_records, columns=polypharm_columns)
                .sort_values('n_targets_hit', ascending=False))
polypharm_path = screen_path / 'cross_target_polypharmacology_hits.csv'
polypharm_df.to_csv(polypharm_path, index=False)
_register('cross_target_polypharmacology_hits.csv', polypharm_path, len(polypharm_df))
print(f'Cross-target hits (full pool, high-confidence, >=2 targets): {len(polypharm_df)} -> {polypharm_path}')
if len(polypharm_df):
    print(polypharm_df.head(10).to_string(index=False))


Cross-target hits (full pool, high-confidence, >=2 targets): 2 -> ml/results/drugbank_screening/cross_target_polypharmacology_hits.csv
global_compound_id drugbank_id generic_name  n_targets_hit targets_hit  mean_calibrated_proba
 CMPD_6c6de4707796     DB12960    INCB-9471              2 CCR5; OPRM1               0.950550
 CMPD_b8c2d366bced     DB17077     AZD-1940              2  CB2; OPRM1               0.914185


## Step 7: Chemical Diversity of Novel Candidates

For the novel-candidate shortlist: Bemis-Murcko scaffold counts and
singleton-scaffold rate (does the shortlist cluster on one chemotype, or
is it genuinely diverse?), Tanimoto similarity of each candidate to its
nearest TRAINING compound (a chemistry-native similarity measure,
distinct from applicability-domain distance which is Euclidean in scaled
descriptor space -- this answers "how close in substructure is this
candidate to something the model actually saw", restricted to the
shortlisted candidates only for compute reasons, not the full DrugBank
library), and a scaffold-based clustering pass so the top-N publication
table shows one representative per chemotype rather than 20 near-identical
analogues.


In [21]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs

def compute_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaf if scaf else smiles
    except Exception:
        return smiles


if len(novel_df):
    with BlockLogs():
        novel_df = novel_df.copy()
        novel_df['murcko_scaffold_smiles'] = novel_df['clean_smiles'].apply(compute_scaffold)
    scaffold_counts = novel_df.groupby('target')['murcko_scaffold_smiles'].agg(['nunique', 'count'])
    scaffold_counts.columns = ['n_scaffolds', 'n_compounds']
    scaffold_counts['scaffold_richness'] = scaffold_counts['n_scaffolds'] / scaffold_counts['n_compounds']
    scaffold_size = novel_df.groupby(['target', 'murcko_scaffold_smiles']).size()
    singleton_pct = (scaffold_size == 1).groupby('target').mean() * 100
    scaffold_counts['pct_singleton_scaffolds'] = singleton_pct
    print('Scaffold diversity of novel candidates, per target:')
    print(scaffold_counts.to_string())
    scaffold_diversity_path = screen_path / 'novel_candidates_scaffold_diversity.csv'
    scaffold_counts.reset_index().to_csv(scaffold_diversity_path, index=False)
    _register('novel_candidates_scaffold_diversity.csv', scaffold_diversity_path, len(scaffold_counts))
else:
    print('No novel candidates to characterise (empty shortlist).')


Scaffold diversity of novel candidates, per target:
         n_scaffolds  n_compounds  scaffold_richness  pct_singleton_scaffolds
target                                                                       
adora2a            3            3           1.000000               100.000000
cb2                9            9           1.000000               100.000000
drd2              21           29           0.724138                76.190476
oprm1             34           51           0.666667                85.294118


In [22]:
# ---- Tanimoto similarity to nearest TRAINING compound, per novel candidate ----
# Restricted to the (small) novel-candidate set, not the full DrugBank library --
# a full cross-product (thousands x thousands, per target) would be needlessly
# expensive when only the shortlisted candidates need this number.
tanimoto_records = []
if len(novel_df):
    for target in TARGETS:
        cand_sub = novel_df[novel_df['target'] == target]
        if len(cand_sub) == 0:
            continue
        for pool in cand_sub['activity_pool'].unique():
            pool_cand = cand_sub[cand_sub['activity_pool'] == pool]
            combo_key = f'{target}_{pool}'
            train_morgan_path = PROJECT_DIR / 'ml' / 'features' / combo_key / 'morgan.npy'
            if not train_morgan_path.exists():
                continue
            X_train_morgan = np.load(train_morgan_path)
            train_bitvects = [
                DataStructs.CreateFromBitString(''.join(str(b) for b in row))
                for row in X_train_morgan.astype(int)
            ]
            cand_mols = smiles_to_mols(pool_cand['clean_smiles'].tolist())
            for (_, row), mol in zip(pool_cand.iterrows(), cand_mols):
                if mol is None:
                    continue
                cand_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS)
                sims = DataStructs.BulkTanimotoSimilarity(cand_fp, train_bitvects)
                tanimoto_records.append({
                    'target': target, 'activity_pool': pool,
                    'global_compound_id': row['global_compound_id'],
                    'max_tanimoto_to_training': float(max(sims)) if sims else np.nan,
                })

tanimoto_df = pd.DataFrame(tanimoto_records)
if len(tanimoto_df):
    novel_df = novel_df.merge(tanimoto_df, on=['target', 'activity_pool', 'global_compound_id'], how='left')
    print(f'Tanimoto-to-nearest-training-compound computed for {len(tanimoto_df)} novel candidates.')
    print(novel_df.groupby('target')['max_tanimoto_to_training'].describe().to_string())
else:
    print('No Tanimoto similarities computed (empty shortlist or no training features found).')


Tanimoto-to-nearest-training-compound computed for 92 novel candidates.
         count      mean       std       min       25%       50%       75%  max
target                                                                         
adora2a    3.0  0.609630  0.346170  0.340000  0.414444  0.488889  0.744444  1.0
cb2        9.0  0.350564  0.247097  0.188889  0.247312  0.275000  0.308511  1.0
drd2      29.0  0.760840  0.217379  0.280899  0.662162  0.772727  1.000000  1.0
oprm1     51.0  0.687168  0.228030  0.231481  0.550876  0.714286  0.827457  1.0


In [23]:
# ---- Deduplicated top-N publication table: one representative per
# Bemis-Murcko scaffold, ranked by priority score, so the top-N table isn't
# 20 near-identical analogues of the same chemotype. ----
if len(novel_df) and 'murcko_scaffold_smiles' in novel_df.columns:
    top_n_records = []
    for target in TARGETS:
        sub = novel_df[novel_df['target'] == target].sort_values('priority_score', ascending=False)
        seen_scaffolds = set()
        for _, row in sub.iterrows():
            scaf = row['murcko_scaffold_smiles']
            if scaf in seen_scaffolds:
                continue
            seen_scaffolds.add(scaf)
            top_n_records.append(row.to_dict())
            if len(seen_scaffolds) >= TOP_N_PUBLICATION_TABLE:
                break
    top_n_df = pd.DataFrame(top_n_records)
    top_n_path = screen_path / 'publication_top_candidates_deduplicated.csv'
    top_n_df.to_csv(top_n_path, index=False)
    _register('publication_top_candidates_deduplicated.csv', top_n_path, len(top_n_df))
    print(f'Publication top-{TOP_N_PUBLICATION_TABLE}-per-target (one per scaffold cluster): '
          f'{len(top_n_df)} rows -> {top_n_path}')
else:
    top_n_df = pd.DataFrame()
    print('No deduplicated top-N table produced (empty shortlist).')

# Re-save the primary ranked-candidate deliverable WITH the Step-7 enrichment
# columns (Murcko scaffold, max Tanimoto to nearest training compound) now that
# they exist, and refresh shortlist_df so the Excel workbook and figures below
# carry them too — Step 5 saved these before enrichment existed.
if len(novel_df):
    ranked_df = novel_df.sort_values('priority_score', ascending=False)
    ranked_df.to_csv(ranked_path, index=False)
    shortlist_df = ranked_df
    print(f'Primary ranked candidates re-saved with scaffold + Tanimoto enrichment -> {ranked_path}')


Publication top-20-per-target (one per scaffold cluster): 52 rows -> ml/results/drugbank_screening/publication_top_candidates_deduplicated.csv
Primary ranked candidates re-saved with scaffold + Tanimoto enrichment -> ml/results/drugbank_screening/drugbank_primary_ranked_candidates.csv


## Step 8: Saved Summary Tables

Persist both layers explicitly so manuscript numbers cannot silently mix provenance:

1. the **primary deployed full/combined screen**, which feeds prospective candidate
   nomination and docking; and
2. the **all-pool combined-representation sensitivity screen**, retained only to
   describe endpoint-pool robustness.

Cross-representation consensus remains a sensitivity analysis and does not alter
primary candidate membership.


In [24]:
# ---- PRIMARY deployed full/combined screening summary: one row per target ----
primary_summary_rows = []
for target in TARGETS:
    sub = primary_df[primary_df['target'] == target]
    if len(sub) == 0:
        continue
    hi = sub[sub['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD]
    novel = hi[~hi['already_in_training_set']]
    rediscovered = hi[hi['already_in_training_set']]
    primary_summary_rows.append({
        'target': target, 'activity_pool': 'full',
        'feature_representation': PRIMARY_REPRESENTATION,
        'classification_algorithm': sub['classification_algorithm'].iloc[0],
        'regression_algorithm': sub['regression_algorithm'].iloc[0],
        'n_screened': len(sub), 'n_high_confidence': len(hi),
        'n_novel_candidates': len(novel),
        'n_rediscovered_known': len(rediscovered),
        'high_confidence_rate_pct': len(hi) / len(sub) * 100,
        'pct_high_conf_within_ad': hi['within_applicability_domain'].mean() * 100 if len(hi) else np.nan,
        'mean_calibrated_proba': float(sub['predicted_proba_calibrated'].mean()),
        'median_calibrated_proba': float(sub['predicted_proba_calibrated'].median()),
        'pct_screened_within_ad': sub['within_applicability_domain'].mean() * 100,
    })
primary_screening_summary_df = pd.DataFrame(primary_summary_rows)
primary_summary_path = screen_path / 'screening_summary_primary_deployed.csv'
primary_screening_summary_df.to_csv(primary_summary_path, index=False)
_register('screening_summary_primary_deployed.csv', primary_summary_path, len(primary_screening_summary_df))
print(f'PRIMARY deployed screening summary saved: {primary_summary_path} ({len(primary_screening_summary_df)} rows)')
if len(primary_screening_summary_df):
    print(primary_screening_summary_df.to_string(index=False))

# ---- SENSITIVITY summary: combined representation across every activity pool ----
sensitivity_summary_rows = []
for target in TARGETS:
    for pool in POOLS:
        sub = sensitivity_combined_df[(sensitivity_combined_df['target'] == target) &
                                      (sensitivity_combined_df['activity_pool'] == pool)]
        if len(sub) == 0:
            continue
        hi = sub[sub['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD]
        novel = hi[~hi['already_in_training_set']]
        rediscovered = hi[hi['already_in_training_set']]
        sensitivity_summary_rows.append({
            'target': target, 'activity_pool': pool,
            'feature_representation': PRIMARY_REPRESENTATION,
            'classification_algorithm': sub['classification_algorithm'].iloc[0],
            'regression_algorithm': sub['regression_algorithm'].iloc[0],
            'n_screened': len(sub), 'n_high_confidence': len(hi),
            'n_novel_candidates': len(novel),
            'n_rediscovered_known': len(rediscovered),
            'high_confidence_rate_pct': len(hi) / len(sub) * 100,
            'pct_high_conf_within_ad': hi['within_applicability_domain'].mean() * 100 if len(hi) else np.nan,
            'mean_calibrated_proba': float(sub['predicted_proba_calibrated'].mean()),
            'median_calibrated_proba': float(sub['predicted_proba_calibrated'].median()),
            'pct_screened_within_ad': sub['within_applicability_domain'].mean() * 100,
        })
screening_summary_df = pd.DataFrame(sensitivity_summary_rows)
screening_summary_path = screen_path / 'screening_summary_by_combination.csv'
screening_summary_df.to_csv(screening_summary_path, index=False)
_register('screening_summary_by_combination.csv', screening_summary_path, len(screening_summary_df))
print(f'SENSITIVITY all-pool summary saved: {screening_summary_path} ({len(screening_summary_df)} rows)')



PRIMARY deployed screening summary saved: ml/results/drugbank_screening/screening_summary_primary_deployed.csv (5 rows)
 target activity_pool feature_representation classification_algorithm regression_algorithm  n_screened  n_high_confidence  n_novel_candidates  n_rediscovered_known  high_confidence_rate_pct  pct_high_conf_within_ad  mean_calibrated_proba  median_calibrated_proba  pct_screened_within_ad
   drd2          full               combined                  XGBoost              XGBoost        9874                 66                  29                    37                  0.668422                90.909091               0.373428                 0.398058               33.522382
    cb2          full               combined                 LightGBM              XGBoost        9874                 22                   9                    13                  0.222807                68.181818               0.319526                 0.196721               33.390723
adora2a          fu

In [25]:
# ---- Drug-group / ATC breakdown of hits (novel candidates AND rediscovered),
# saved rather than only printed -- feeds the DrugBank enrichment/status
# analysis. Long format: one row per (hit_class, target, category, value). ----
breakdown_rows = []
for hit_class, hit_df in [('novel_candidate', novel_df), ('rediscovered_known', rediscovery_df)]:
    if len(hit_df) == 0:
        continue
    for category_col in ['drug_groups', 'atc_codes']:
        if category_col not in hit_df.columns:
            continue
        for target in TARGETS:
            t_sub = hit_df[hit_df['target'] == target]
            if len(t_sub) == 0:
                continue
            for value, count in t_sub[category_col].value_counts().items():
                breakdown_rows.append({
                    'hit_class': hit_class, 'target': target, 'category': category_col,
                    'value': value, 'n_hits': int(count),
                    'pct_of_target_hits': count / len(t_sub) * 100,
                })

hit_breakdown_df = pd.DataFrame(breakdown_rows)
hit_breakdown_path = screen_path / 'hit_drug_group_atc_breakdown.csv'
hit_breakdown_df.to_csv(hit_breakdown_path, index=False)
_register('hit_drug_group_atc_breakdown.csv', hit_breakdown_path, len(hit_breakdown_df))
print(f'Drug-group / ATC breakdown of hits saved: {hit_breakdown_path} ({len(hit_breakdown_df)} rows)')


Drug-group / ATC breakdown of hits saved: ml/results/drugbank_screening/hit_drug_group_atc_breakdown.csv (147 rows)


In [26]:
# ---- Standalone Tanimoto-to-training table: saved for EVERY novel candidate
# it was computed for, independent of whether the compound reached a ranked
# file (tanimoto_df from Step 7, persisted directly here). ----
tanimoto_out_path = screen_path / 'novel_candidates_tanimoto_to_training.csv'
tanimoto_df.to_csv(tanimoto_out_path, index=False)
_register('novel_candidates_tanimoto_to_training.csv', tanimoto_out_path, len(tanimoto_df))
print(f'Tanimoto-to-training table saved: {tanimoto_out_path} ({len(tanimoto_df)} rows)')


Tanimoto-to-training table saved: ml/results/drugbank_screening/novel_candidates_tanimoto_to_training.csv (92 rows)


In [27]:
# ---- SENSITIVITY ONLY — Cross-representation consensus: for each compound x (target, pool),
# do the combined / Morgan / descriptors models agree it is a high-confidence
# hit? This is the point of having screened all 3 representations -- a
# candidate flagged by all 3 is more robust than one flagged only by
# 'combined'. Uses screening_results_all.csv (all representations). ----
conf_flag = screening_all_df[~((screening_all_df['target'] == 'ccr5') & (screening_all_df['activity_pool'] == 'ki'))].copy()
conf_flag['is_high_confidence'] = conf_flag['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD
consensus_df = (conf_flag.groupby(['target', 'activity_pool', 'global_compound_id', 'drugbank_id', 'generic_name'])
                .agg(n_representations_high_confidence=('is_high_confidence', 'sum'),
                     mean_calibrated_proba_across_reps=('predicted_proba_calibrated', 'mean'),
                     already_in_training_set=('already_in_training_set', 'first'))
                .reset_index())
# Only keep compounds flagged by at least one representation -- the vast
# majority scored low by all three and carry no consensus signal.
consensus_df = consensus_df[consensus_df['n_representations_high_confidence'] >= 1].copy()
consensus_df['unanimous_across_representations'] = consensus_df['n_representations_high_confidence'] == len(FEATURE_REPS)
consensus_df = consensus_df.sort_values(
    ['n_representations_high_confidence', 'mean_calibrated_proba_across_reps'], ascending=False)

consensus_path = screen_path / 'cross_representation_consensus.csv'
consensus_df.to_csv(consensus_path, index=False)
_register('cross_representation_consensus.csv', consensus_path, len(consensus_df))
n_unanimous = int(consensus_df['unanimous_across_representations'].sum()) if len(consensus_df) else 0
print(f'Cross-representation consensus saved: {consensus_path} ({len(consensus_df)} compound-combinations '
      f'flagged by >=1 representation; {n_unanimous} unanimous across all {len(FEATURE_REPS)})')


Cross-representation consensus saved: ml/results/drugbank_screening/cross_representation_consensus.csv (4695 compound-combinations flagged by >=1 representation; 198 unanimous across all 3)


## Figures

In [28]:
print("=== CALIBRATED PROBABILITY DISTRIBUTION (primary representation, full pool) ===")

full_primary = primary_df.copy()  # primary_df is already full-pool + combined only
fig, ax = plt.subplots(figsize=(7, 5))
for target in TARGETS:
    sub = full_primary[full_primary['target'] == target]
    ax.hist(sub['predicted_proba_calibrated'], bins=30, histtype='step', color=TARGET_COLORS[target],
            label=target.upper(), linewidth=1.5)
ax.axvline(HIGH_CONFIDENCE_THRESHOLD, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Calibrated probability')
ax.set_ylabel('DrugBank compound count')
ax.legend(frameon=False, fontsize=8)
panel_label(ax, 'A')
fig.tight_layout()
save_fig(fig, 'figure_drugbank_probability_distribution')

full_primary[['target', 'drugbank_id', 'predicted_proba_calibrated']].to_csv(
    fig_dir / 'figure_drugbank_probability_distribution_data.csv', index=False)
_register('figure_drugbank_probability_distribution_data.csv',
          fig_dir / 'figure_drugbank_probability_distribution_data.csv', len(full_primary))
print('Figure saved: figure_drugbank_probability_distribution (+ plot-source CSV)')


=== CALIBRATED PROBABILITY DISTRIBUTION (primary representation, full pool) ===
Figure saved: figure_drugbank_probability_distribution (+ plot-source CSV)


In [29]:
print("=== SHORTLIST COUNTS AND APPLICABILITY-DOMAIN COVERAGE PER TARGET ===")

if len(shortlist_df):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    counts = shortlist_df.groupby('target').size().reindex(TARGETS).fillna(0)
    axes[0].bar(range(len(TARGETS)), counts.values, color=[TARGET_COLORS[t] for t in TARGETS])
    axes[0].set_xticks(range(len(TARGETS))); axes[0].set_xticklabels([t.upper() for t in TARGETS])
    axes[0].set_ylabel('Novel high-confidence candidates')
    panel_label(axes[0], 'A')

    ad_pct = shortlist_df.groupby('target')['within_applicability_domain'].mean().reindex(TARGETS).fillna(0) * 100
    axes[1].bar(range(len(TARGETS)), ad_pct.values, color=[TARGET_COLORS[t] for t in TARGETS])
    axes[1].set_xticks(range(len(TARGETS))); axes[1].set_xticklabels([t.upper() for t in TARGETS])
    axes[1].set_ylabel('Candidates within applicability domain (%)')
    panel_label(axes[1], 'B')
    fig.tight_layout()
    save_fig(fig, 'figure_shortlist_summary')

    shortlist_df.groupby('target').agg(
        n_hits=('drugbank_id', 'size'),
        pct_within_ad=('within_applicability_domain', 'mean'),
    ).to_csv(fig_dir / 'figure_shortlist_summary_data.csv')
    _register('figure_shortlist_summary_data.csv', fig_dir / 'figure_shortlist_summary_data.csv')
    print('Figure saved: figure_shortlist_summary (+ plot-source CSV)')
else:
    print('No shortlisted hits — figure skipped.')


=== SHORTLIST COUNTS AND APPLICABILITY-DOMAIN COVERAGE PER TARGET ===
Figure saved: figure_shortlist_summary (+ plot-source CSV)


In [30]:
print("=== UNCERTAINTY-VS-APPLICABILITY-DOMAIN PANEL (full pool, primary representation) ===")

unc_df = full_primary.copy()
unc_df['conformal_interval_width'] = unc_df['pactivity_interval_hi_90pct'] - unc_df['pactivity_interval_lo_90pct']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# A: calibrated probability vs AD distance
ax = axes[0, 0]
for target in TARGETS:
    sub = unc_df[unc_df['target'] == target]
    ax.scatter(sub['applicability_domain_distance'], sub['predicted_proba_calibrated'],
               s=4, alpha=0.25, color=TARGET_COLORS[target], label=target.upper())
ax.set_xlabel('Applicability-domain distance')
ax.set_ylabel('Calibrated probability')
panel_label(ax, 'A')

# B: predicted pActivity vs conformal interval width
ax = axes[0, 1]
for target in TARGETS:
    sub = unc_df[unc_df['target'] == target].dropna(subset=['predicted_pactivity', 'conformal_interval_width'])
    ax.scatter(sub['predicted_pactivity'], sub['conformal_interval_width'],
               s=4, alpha=0.25, color=TARGET_COLORS[target])
ax.set_xlabel('Predicted pActivity')
ax.set_ylabel('Conformal interval width (90%)')
panel_label(ax, 'B')

# C: hit count inside vs outside AD (high-confidence hits only)
ax = axes[0, 2]
hi_conf = unc_df[unc_df['predicted_proba_calibrated'] >= HIGH_CONFIDENCE_THRESHOLD]
ad_counts = hi_conf.groupby(['target', 'within_applicability_domain']).size().unstack(fill_value=0)
ad_counts = ad_counts.reindex(TARGETS).fillna(0)
x = np.arange(len(TARGETS))
if True in ad_counts.columns:
    ax.bar(x - 0.2, ad_counts.get(True, 0), 0.4, label='Within AD', color='#0072B2')
if False in ad_counts.columns:
    ax.bar(x + 0.2, ad_counts.get(False, 0), 0.4, label='Outside AD', color='#D55E00')
ax.set_xticks(x); ax.set_xticklabels([t.upper() for t in TARGETS])
ax.set_ylabel('High-confidence hit count')
ax.legend(frameon=False, fontsize=8)
panel_label(ax, 'C')

# D: probability histogram split by AD status
ax = axes[1, 0]
ax.hist(unc_df.loc[unc_df['within_applicability_domain'], 'predicted_proba_calibrated'],
        bins=30, histtype='step', color='#0072B2', label='Within AD', linewidth=1.5)
ax.hist(unc_df.loc[~unc_df['within_applicability_domain'], 'predicted_proba_calibrated'],
        bins=30, histtype='step', color='#D55E00', label='Outside AD', linewidth=1.5)
ax.axvline(HIGH_CONFIDENCE_THRESHOLD, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Calibrated probability')
ax.set_ylabel('DrugBank compound count')
ax.legend(frameon=False, fontsize=8)
panel_label(ax, 'D')

# E: conformal set-size distribution
ax = axes[1, 1]
set_size_counts = unc_df.groupby(['target', 'conformal_set_size_90pct']).size().unstack(fill_value=0)
set_size_counts = set_size_counts.reindex(TARGETS).fillna(0)
bottom = np.zeros(len(TARGETS))
for size_val, color in [(1, '#0072B2'), (2, '#D55E00'), (0, '#999999')]:
    if size_val in set_size_counts.columns:
        vals = set_size_counts[size_val].values
        ax.bar(x, vals, bottom=bottom, label=f'set size {int(size_val)}', color=color)
        bottom += vals
ax.set_xticks(x); ax.set_xticklabels([t.upper() for t in TARGETS])
ax.set_ylabel('DrugBank compound count')
ax.legend(frameon=False, fontsize=8)
panel_label(ax, 'E')

axes[1, 2].axis('off')

fig.tight_layout()
save_fig(fig, 'figure_uncertainty_vs_applicability_domain')

unc_df[['target', 'drugbank_id', 'predicted_proba_calibrated', 'applicability_domain_distance',
        'within_applicability_domain', 'predicted_pactivity', 'conformal_interval_width',
        'conformal_set_size_90pct']].to_csv(
    fig_dir / 'figure_uncertainty_vs_applicability_domain_data.csv', index=False)
_register('figure_uncertainty_vs_applicability_domain_data.csv',
          fig_dir / 'figure_uncertainty_vs_applicability_domain_data.csv', len(unc_df))
print('Figure saved: figure_uncertainty_vs_applicability_domain (5 panels, + plot-source CSV)')


=== UNCERTAINTY-VS-APPLICABILITY-DOMAIN PANEL (full pool, primary representation) ===
Figure saved: figure_uncertainty_vs_applicability_domain (5 panels, + plot-source CSV)


## Final Deliverables

Consolidated exports: the full per-representation predictions table
(already saved as `screening_results_all.csv`), the two novel-candidate
rankings, the known-rediscovery sanity check, the cross-target
polypharmacology table, and a per-target multi-sheet Excel workbook plus
a publication-ready top-10/top-20 table for direct manuscript use.


In [31]:
# ---- Multi-sheet Excel workbook: one sheet per target, novel candidates
# ranked by priority score (falls back to an empty placeholder sheet for
# any target with zero shortlisted hits, so the workbook always has all 5
# target sheets, not just the ones with hits). ----
excel_path = screen_path / 'drugbank_top_candidates_per_target.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for target in TARGETS:
        sub = shortlist_df[shortlist_df['target'] == target] if len(shortlist_df) else pd.DataFrame()
        if len(sub) == 0:
            sub = pd.DataFrame({'note': [f'No high-confidence novel candidates for {target.upper()}']})
        sub.to_excel(writer, sheet_name=target.upper()[:31], index=False)
_register('drugbank_top_candidates_per_target.xlsx', excel_path)
print(f'Per-target Excel workbook saved: {excel_path}')

# ---- Publication-ready top-10 and top-20 tables (deduplicated by scaffold,
# already computed in Step 7 as top_n_df) ----
pub_cols = ['target', 'activity_pool', 'drugbank_id', 'generic_name', 'drug_groups',
            'predicted_proba_calibrated', 'priority_score', 'within_applicability_domain',
            'predicted_pactivity', 'max_tanimoto_to_training']
pub_cols_present = [c for c in pub_cols if c in top_n_df.columns] if len(top_n_df) else []

if len(top_n_df):
    top10_df = (top_n_df.sort_values('priority_score', ascending=False)
                .groupby('target').head(10)[pub_cols_present])
    top20_df = top_n_df[pub_cols_present]
else:
    top10_df = pd.DataFrame(columns=pub_cols)
    top20_df = pd.DataFrame(columns=pub_cols)

top10_path = screen_path / 'publication_table_top10_per_target.csv'
top20_path = screen_path / 'publication_table_top20_per_target.csv'
top10_df.to_csv(top10_path, index=False)
top20_df.to_csv(top20_path, index=False)
_register('publication_table_top10_per_target.csv', top10_path, len(top10_df))
_register('publication_table_top20_per_target.csv', top20_path, len(top20_df))
print(f'Publication tables saved: {top10_path.name} ({len(top10_df)} rows), {top20_path.name} ({len(top20_df)} rows)')


Per-target Excel workbook saved: ml/results/drugbank_screening/drugbank_top_candidates_per_target.xlsx
Publication tables saved: publication_table_top10_per_target.csv (32 rows), publication_table_top20_per_target.csv (52 rows)


## Final Manifest

In [32]:
FULL_CONFIG = {
    'targets': TARGETS,
    'pools': POOLS,
    'feature_representations': FEATURE_REPS,
    'primary_representation': PRIMARY_REPRESENTATION,
    'primary_activity_pool': 'full',
    'primary_screening_definition': 'deployed full-pool + combined-representation model per target',
    'sensitivity_screening_definition': 'all activity pools and feature representations; not used for primary candidate nomination',
    'algorithms': ALGORITHMS,
    'morgan_radius': MORGAN_RADIUS,
    'morgan_n_bits': MORGAN_NBITS,
    'descriptor_columns': DESCRIPTOR_COLS,
    'high_confidence_threshold': HIGH_CONFIDENCE_THRESHOLD,
    'drugbank_sdf_path': str(DRUGBANK_SDF_PATH),
    'drugbank_full_xml_path': str(DRUGBANK_FULL_XML_PATH) if DRUGBANK_FULL_XML_PATH else None,
    'n_drugbank_compounds_standardised': len(drugbank_df),
    'priority_score_weights': PRIORITY_WEIGHTS,
    'top_n_publication_table': TOP_N_PUBLICATION_TABLE,
    'random_seed': RANDOM_STATE,
}

manifest_inputs = {
    'best_algorithm_by_combination.csv': {'path': str(best_algo_path)},
    'manifest_03_ml_benchmark.json': {'path': str(manifest_03_path)},  # anchors every nb3 artifact hash
    'drugbank_sdf': {'path': str(DRUGBANK_SDF_PATH)},
}
if DRUGBANK_FULL_XML_PATH is not None and Path(DRUGBANK_FULL_XML_PATH).exists():
    manifest_inputs['drugbank_full_xml'] = {'path': str(DRUGBANK_FULL_XML_PATH)}

# Every notebook-3 model artifact this screening run actually LOADED (deployed
# model + calibrator + conformal wrapper + AD reference, per target/pool/
# representation and the deployment-selected classifier/regressor) — so the
# manifest hashes the exact model files predictions were produced from, not
# just the CSV that named which algorithm to use.
for target in TARGETS:
    for pool in POOLS:
        paths = get_combo_paths(target, pool)
        if paths['cleaned_data_path'].exists():
            manifest_inputs[f'cleaned_data_{target}_{pool}.csv'] = {'path': str(paths['cleaned_data_path'])}
        clf_key = get_best_algo_key(target, pool, 'classification')
        reg_key = get_best_algo_key(target, pool, 'regression')
        for representation in FEATURE_REPS:
            rep_dir = paths['models_dir'] / representation
            candidate_artifacts = ['ad_reference.pkl']
            if clf_key is not None:
                candidate_artifacts += [f'{clf_key}_clf.joblib', f'{clf_key}_clf_calibrator.pkl',
                                        f'{clf_key}_clf_conformal.pkl']
            if reg_key is not None:
                candidate_artifacts += [f'{reg_key}_reg.joblib', f'{reg_key}_reg_conformal.pkl']
            for artifact in candidate_artifacts:
                apath = rep_dir / artifact
                if apath.exists():
                    manifest_inputs[f'{target}_{pool}/{representation}/{artifact}'] = {'path': str(apath)}

write_manifest(
    screen_path / 'manifest_04_drugbank_screening.json',
    config_summary=FULL_CONFIG,
    outputs=all_outputs,
    inputs=manifest_inputs,
)


Manifest saved: ml/results/drugbank_screening/manifest_04_drugbank_screening.json (289 inputs, 73 outputs hashed)


{'timestamp': '2026-08-31T09:29:50.920393',
 'python_version': '3.11.13',
 'git_commit': None,
 'config': {'targets': ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5'],
  'pools': ['ki', 'ki_ic50', 'full'],
  'feature_representations': ['morgan', 'descriptors', 'combined'],
  'primary_representation': 'combined',
  'primary_activity_pool': 'full',
  'primary_screening_definition': 'deployed full-pool + combined-representation model per target',
  'sensitivity_screening_definition': 'all activity pools and feature representations; not used for primary candidate nomination',
  'algorithms': ['rf', 'xgb', 'lgb'],
  'morgan_radius': 2,
  'morgan_n_bits': 2048,
  'descriptor_columns': ['MW',
   'LogP',
   'TPSA',
   'HBD',
   'HBA',
   'RotBonds',
   'HeavyAtomCount',
   'RingCount',
   'AromaticRingCount',
   'FractionCSP3'],
  'high_confidence_threshold': 0.9,
  'drugbank_sdf_path': 'data/external/drugbank/drugbank.sdf',
  'drugbank_full_xml_path': 'data/external/drugbank/drugbank_full_database

## Summary

In [33]:
print('DrugBank screening complete.')
print(f'\nDrugBank library: {len(drugbank_df)} standardised compounds screened across '
      f'{len(TARGETS)} targets x {len(POOLS)} pools x {len(FEATURE_REPS)} representations for sensitivity analysis.')
print('PRIMARY prospective candidate nomination: deployed FULL-pool + COMBINED-representation model per target only.')
print(f'Total all-model sensitivity rows: {len(screening_all_df):,}')
print(f'Primary full/combined screening rows: {len(primary_df):,}')
print(f'Known rediscovery (primary screen; high-confidence, already in training): {len(rediscovery_df)}')
print(f'Novel candidates (primary screen; high-confidence, not in training): {len(novel_df)}')
print('  -> ranked by calibrated probability AND composite priority score')
print(f'Deduplicated publication top-{TOP_N_PUBLICATION_TABLE}-per-target: {len(top_n_df)}')
print(f'Cross-target polypharmacology hits (primary full/combined screen, >=2 targets): {len(polypharm_df)}')
print('\nPrimary deliverables:')
print('  drugbank_known_rediscovery.csv')
print('  drugbank_novel_candidates_ranked_by_probability.csv')
print('  drugbank_primary_ranked_candidates.csv')
print('  screening_summary_primary_deployed.csv')
print('  publication_table_top10_per_target.csv / publication_table_top20_per_target.csv')
print('\nSensitivity/supporting deliverables:')
print('  screening_results_all.csv')
print('  screening_summary_by_combination.csv')
print('  cross_representation_consensus.csv')
print(f'\nAll outputs: {screen_path}')
print(f'Manifest: {screen_path / "manifest_04_drugbank_screening.json"}')



DrugBank screening complete.

DrugBank library: 9874 standardised compounds screened across 5 targets x 3 pools x 3 representations for sensitivity analysis.
PRIMARY prospective candidate nomination: deployed FULL-pool + COMBINED-representation model per target only.
Total all-model sensitivity rows: 444,330
Primary full/combined screening rows: 49,370
Known rediscovery (primary screen; high-confidence, already in training): 91
Novel candidates (primary screen; high-confidence, not in training): 92
  -> ranked by calibrated probability AND composite priority score
Deduplicated publication top-20-per-target: 52
Cross-target polypharmacology hits (primary full/combined screen, >=2 targets): 2

Primary deliverables:
  drugbank_known_rediscovery.csv
  drugbank_novel_candidates_ranked_by_probability.csv
  drugbank_primary_ranked_candidates.csv
  screening_summary_primary_deployed.csv
  publication_table_top10_per_target.csv / publication_table_top20_per_target.csv

Sensitivity/supporting de